# 03.02 — Ingesta de datos meteorológicos NOAA

## Objetivo

En este notebook se implementará la ingesta de información meteorológica
proveniente de la API Climate Data Online (CDO) de NOAA.

A diferencia de los datos de NYC Taxi, que se obtuvieron como archivos
Parquet, esta fuente será consumida mediante una API REST que devuelve
información en formato JSON.

Durante el notebook se realizará:

1. Configuración de la conexión con la API de NOAA.
2. Validación de la estación meteorológica seleccionada.
3. Consulta de variables meteorológicas para 2024.
4. Implementación de paginación.
5. Almacenamiento de las respuestas originales en la capa `raw`.
6. Lectura de los archivos JSON mediante PySpark.
7. Validación inicial del esquema, período y variables disponibles.

La estación utilizada será NY City Central Park, identificada por NOAA como
`GHCND:USW00094728`.

## 1. Importaciones

In [1]:
import json
import os

from getpass import getpass
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## 2. Sesión de Spark

In [2]:
spark = (
    SparkSession.builder
    .appName("03_02_noaa_weather_ingestion")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [3]:
print("=" * 60)
print("INFORMACIÓN DEL ENTORNO SPARK")
print("=" * 60)

print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

INFORMACIÓN DEL ENTORNO SPARK
Spark version: 4.1.3
Application name: 03_02_noaa_weather_ingestion
Master: local[*]
Default parallelism: 12


## 3. Configuración de la fuente NOAA

NOAA proporciona acceso programático a Climate Data Online mediante una API
REST.

Para este proyecto se utilizará el dataset **GHCND (Global Historical
Climatology Network Daily)**, que contiene observaciones meteorológicas diarias.

La estación seleccionada es **NY City Central Park**, que servirá como
aproximación de las condiciones meteorológicas generales de Nueva York durante
2024.

El dataset Daily Summaries incluye observaciones meteorológicas diarias y NOAA permite consultar una estación concreta mediante el endpoint /data.


Configuramos:

In [4]:
YEAR = 2024

DATASET_ID = "GHCND"
STATION_ID = "GHCND:USW00094728"

START_DATE = f"{YEAR}-01-01"
END_DATE = f"{YEAR}-12-31"

BASE_API_URL = "https://www.ncei.noaa.gov/cdo-web/api/v2"

## 4. Variables meteorológicas

| Código | Variable |
|---|---|
| `TMAX` | temperatura máxima diaria |
| `TMIN` | temperatura mínima diaria |
| `PRCP` | precipitación |
| `SNOW` | nieve |
| `AWND` | velocidad promedio del viento |

In [5]:
WEATHER_VARIABLES = [
    "TMAX",
    "TMIN",
    "PRCP",
    "SNOW",
    "AWND"
]

Son particularmente interesantes para nuestro problema porque después podremos investigar relaciones como:

```text
lluvia
  ↓
demanda de taxi

nieve
  ↓
duración del viaje

temperatura
  ↓
cantidad de viajes

viento / condiciones meteorológicas
  ↓
velocidad y duración
```

## 5. Configurar el directorio `raw`

In [6]:
PROJECT_ROOT = Path.cwd().parent

RAW_WEATHER_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "weather"
    / f"year={YEAR}"
    / "station=USW00094728"
)

RAW_WEATHER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Directorio raw: {RAW_WEATHER_DIR}")

Directorio raw: c:\Users\derec\Desktop\Proyectos\BigData_project\data\raw\weather\year=2024\station=USW00094728


Queremos terminar con algo como:

```text
data/
└── raw/
    ├── taxi/
    │   └── year=2024/
    │
    └── weather/
        └── year=2024/
            └── station=USW00094728/
                ├── page_001.json
                ├── page_002.json
                └── ...
```

## 6. Token de NOAA

No vamos a escribir el token directamente en el notebook.

NOAA exige un token para consumir esta API. El acceso es gratuito, pero requiere registrarse y recibir la clave por correo.

Puedes registrarte desde Climate Data Online:

https://www.ncei.noaa.gov/cdo-web

Cuando ejecutes la celda aparecerá: `Introduce tu token de NOAA` pero **el valor no aparecerá escrito en el notebook.**


Para acceder a la API de Climate Data Online (CDO) v2 de la NOAA, no se requiere un formulario de registro complejo ni crear una contraseña. 

Solo necesitas solicitar un Token de acceso web (clave de API) gratuito.

- Paso 1: Solicitar tu Token de API

    - Ingresa al portal oficial de solicitudes de tokens:👉 https://www.ncdc.noaa.gov/cdo-web/token   
    - Escribe tu correo electrónico en el campo indicado.   
    - Haz clic en el botón Submit.   
    - Revisa tu bandeja de entrada (y la carpeta de Spam o Correo no deseado). Recibirás un mensaje automático de la NOAA con tu Token (un código alfanumérico único).

In [7]:
NOAA_TOKEN = os.getenv("NOAA_TOKEN")

if not NOAA_TOKEN:
    NOAA_TOKEN = getpass(
        "Introduce tu token de NOAA: "
    )

print("Token cargado correctamente.")

Token cargado correctamente.


## 7. Crear una función genérica para consultar NOAA

In [ ]:
def request_noaa_json(endpoint: str, params: dict | None = None) -> dict:
    """
    Realiza una petición GET a la API de NOAA CDO.

    Parameters
    ----------
    endpoint : str
        Endpoint relativo de la API.

    params : dict, optional
        Parámetros query de la petición.

    Returns
    -------
    dict
        Respuesta JSON convertida a diccionario Python.
    """

    url = f"{BASE_API_URL}/{endpoint}"

    if params:
        query_string = urlencode(
            params,
            doseq=True      # lo necesitaremos porque tendremos parámetros con múltiples valores
        )

        url = f"{url}?{query_string}"

    request = Request(
        url,
        headers={
            "token": NOAA_TOKEN
        }
    )

    try:

        with urlopen(request) as response:

            payload = json.loads(
                response.read().decode("utf-8")
            )

        return payload

    except HTTPError as error:

        raise RuntimeError(
            f"Error HTTP {error.code}: {error.reason}"
        ) from error

    except URLError as error:

        raise RuntimeError(
            f"Error de conexión con NOAA: {error.reason}"
        ) from error

## 8. Validar la estación

Antes de solicitar datos meteorológicos quiero que validemos programáticamente que estamos usando la estación que creemos estar usando.

Según NOAA, deberíamos encontrar información de NY City Central Park, con identificador GHCND:USW00094728, aproximadamente en latitud 40.77898, longitud -73.96925 y elevación 42.7 m.

In [9]:
station_info = request_noaa_json(
    endpoint=f"stations/{STATION_ID}"
)

station_info

{'elevation': 42.7,
 'mindate': '1869-01-01',
 'maxdate': '2026-09-24',
 'latitude': 40.77898,
 'name': 'NY CITY CENTRAL PARK, NY US',
 'datacoverage': 1,
 'id': 'GHCND:USW00094728',
 'elevationUnit': 'METERS',
 'longitude': -73.96925}

In [10]:
print("=" * 60)
print("ESTACIÓN METEOROLÓGICA")
print("=" * 60)

print(f"ID:        {station_info.get('id')}")
print(f"Nombre:    {station_info.get('name')}")
print(f"Latitud:   {station_info.get('latitude')}")
print(f"Longitud:  {station_info.get('longitude')}")
print(f"Elevación: {station_info.get('elevation')}")
print(f"Inicio:    {station_info.get('mindate')}")
print(f"Fin:       {station_info.get('maxdate')}")

ESTACIÓN METEOROLÓGICA
ID:        GHCND:USW00094728
Nombre:    NY CITY CENTRAL PARK, NY US
Latitud:   40.77898
Longitud:  -73.96925
Elevación: 42.7
Inicio:    1869-01-01
Fin:       2026-09-24


## 9. Primera consulta de prueba

Igual que hicimos con Taxi, solicitemos únicamente enero.


In [11]:
test_params = {
    "datasetid": DATASET_ID,
    "stationid": STATION_ID,
    "datatypeid": WEATHER_VARIABLES,
    "startdate": "2024-01-01",
    "enddate": "2024-01-31",
    "units": "metric",
    "limit": 1000
}

Es importante que aparezca:

```python
"units": "metric"
```

porque NOAA indica que con este parámetro realiza el escalado y conversión de las observaciones a unidades métricas; si no se proporciona una unidad, los valores se devuelven sin ese escalado/conversión.

Ahora hacemos la consulta:

In [12]:
test_response = request_noaa_json(
    endpoint="data",
    params=test_params
)

vemos las claves:

In [13]:
test_response.keys()

dict_keys(['metadata', 'results'])

Esperamos algo similar a:

```python
dict_keys(["metadata", "results"])
```

## 10. Inspeccionar la metadata

In [14]:
test_response["metadata"]

{'resultset': {'offset': 1, 'count': 153, 'limit': 1000}}

y especificamente:

In [15]:
resultset = (
    test_response
    .get("metadata", {})
    .get("resultset", {})
)

print(f"Resultados totales: {resultset.get('count')}")
print(f"Límite de página:   {resultset.get('limit')}")
print(f"Offset:             {resultset.get('offset')}")

Resultados totales: 153
Límite de página:   1000
Offset:             1


Aquí introducimos otro concepto importante:

### Paginación

La API no necesariamente devuelve todos los datos en una única respuesta.
NOAA admite un máximo de 1,000 resultados por petición, por lo que para consultas grandes debemos iterar mediante offset.

Esto hará que nuestro pipeline sea: 

```text
Request 1
offset=1
limit=1000
       │
       ▼
primeros 1,000 registros

Request 2
offset=1001
limit=1000
       │
       ▼
siguientes 1,000

...

```

Para enero probablemente una página sea suficiente.

Para todo 2024 con cinco variables seguramente necesitaremos más de una.

## 11. Examinar los primeros registros

In [16]:
test_results = test_response.get(
    "results",
    []
)

print(
    f"Registros recibidos: "
    f"{len(test_results)}"
)

Registros recibidos: 153


In [17]:
for record in test_results[:10]:
    print(record)

{'date': '2024-01-01T00:00:00', 'datatype': 'AWND', 'station': 'GHCND:USW00094728', 'attributes': ',,W,', 'value': 1.5}
{'date': '2024-01-01T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:USW00094728', 'attributes': ',,W,2400', 'value': 0.8}
{'date': '2024-01-01T00:00:00', 'datatype': 'SNOW', 'station': 'GHCND:USW00094728', 'attributes': ',,W,', 'value': 0.0}
{'date': '2024-01-01T00:00:00', 'datatype': 'TMAX', 'station': 'GHCND:USW00094728', 'attributes': ',,W,2400', 'value': 8.3}
{'date': '2024-01-01T00:00:00', 'datatype': 'TMIN', 'station': 'GHCND:USW00094728', 'attributes': ',,W,2400', 'value': 1.7}
{'date': '2024-01-02T00:00:00', 'datatype': 'AWND', 'station': 'GHCND:USW00094728', 'attributes': ',,W,', 'value': 1.8}
{'date': '2024-01-02T00:00:00', 'datatype': 'PRCP', 'station': 'GHCND:USW00094728', 'attributes': ',,W,2400', 'value': 0.0}
{'date': '2024-01-02T00:00:00', 'datatype': 'SNOW', 'station': 'GHCND:USW00094728', 'attributes': ',,W,', 'value': 0.0}
{'date': '2024-01-02T00:

Deberías obtener estructuras similares a:

```text
{
    "date": "...",
    "datatype": "TMAX",
    "station": "GHCND:USW00094728",
    "attributes": "...",
    "value": ...
}
```

Esto revela algo muy importante sobre la estructura de NOAA.

Los datos vienen en formato largo:

```text
date        datatype    value
──────────────────────────────
2024-01-01  TMAX        ...
2024-01-01  TMIN        ...
2024-01-01  PRCP        ...
2024-01-01  SNOW        ...
2024-01-01  AWND        ...
2024-01-02  TMAX        ...
```

## 12. Función de ingesta paginada

In [18]:
def download_noaa_weather(
    year: int,
    station_id: str,
    datatypes: list[str],
    output_dir: Path,
    limit: int = 1000
) -> list[Path]:
    """
    Descarga observaciones meteorológicas desde NOAA utilizando
    paginación y conserva cada respuesta JSON en la capa raw.

    Parameters
    ----------
    year : int
        Año que se desea descargar.

    station_id : str
        Identificador NOAA de la estación.

    datatypes : list[str]
        Variables meteorológicas solicitadas.

    output_dir : Path
        Directorio de salida.

    limit : int, default=1000
        Número máximo de resultados por petición.

    Returns
    -------
    list[Path]
        Archivos JSON almacenados.
    """

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    downloaded_files = []

    offset = 1
    page = 1

    while True:

        output_file = (
            output_dir
            / f"page_{page:03d}.json"
        )

        if output_file.exists():

            print(
                f"[SKIP] {output_file.name} ya existe."
            )

            with output_file.open(
                "r",
                encoding="utf-8"
            ) as file:

                payload = json.load(file)

        else:

            params = {
                "datasetid": DATASET_ID,
                "stationid": station_id,
                "datatypeid": datatypes,
                "startdate": f"{year}-01-01",
                "enddate": f"{year}-12-31",
                "units": "metric",
                "limit": limit,
                "offset": offset
            }

            print(
                f"[DOWNLOAD] Página {page} "
                f"(offset={offset})"
            )

            payload = request_noaa_json(
                endpoint="data",
                params=params
            )

            with output_file.open(
                "w",
                encoding="utf-8"
            ) as file:

                json.dump(
                    payload,
                    file,
                    ensure_ascii=False,
                    indent=2
                )

        downloaded_files.append(
            output_file
        )

        results = payload.get(
            "results",
            []
        )

        metadata = (
            payload
            .get("metadata", {})
            .get("resultset", {})
        )

        total_results = metadata.get(
            "count",
            len(results)
        )

        print(
            f"           registros página: {len(results):,} "
            f"| total esperado: {total_results:,}"
        )

        if offset + len(results) > total_results:
            break

        if not results:
            break

        offset += limit
        page += 1

    return downloaded_files

Esta función ya incorpora tres cosas interesantes:

```text
Paginación
+
Persistencia raw
+
Idempotencia
```

## 13. Ejecutar la ingesta

In [19]:
weather_files = download_noaa_weather(
    year=YEAR,
    station_id=STATION_ID,
    datatypes=WEATHER_VARIABLES,
    output_dir=RAW_WEATHER_DIR
)

[DOWNLOAD] Página 1 (offset=1)
           registros página: 1,000 | total esperado: 1,826
[DOWNLOAD] Página 2 (offset=1001)
           registros página: 826 | total esperado: 1,826


## 14. Validar los archivos

In [20]:
print("=" * 60)
print("ARCHIVOS NOAA DESCARGADOS")
print("=" * 60)

for file_path in weather_files:

    size_kb = (
        file_path.stat().st_size
        / 1024
    )

    print(
        f"{file_path.name:<20} "
        f"{size_kb:>10.2f} KB"
    )

ARCHIVOS NOAA DESCARGADOS
page_001.json            166.87 KB
page_002.json            137.88 KB


In [21]:
print(
    f"\nNúmero de páginas: "
    f"{len(weather_files)}"
)


Número de páginas: 2


## 15. Leer los JSON con Spark

Aquí aparece una diferencia importante respecto a Taxi.

Nuestros JSON originales tienen esta estructura:

```text
{
    metadata: {...},
    results: [
        {...},
        {...},
        {...}
    ]
}
```

Por tanto, cada archivo contiene un objeto JSON multilínea, no JSON Lines.

Usamos:

In [22]:
weather_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(str(RAW_WEATHER_DIR))
)

In [23]:
weather_raw_df.printSchema()

root
 |-- metadata: struct (nullable = true)
 |    |-- resultset: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- limit: long (nullable = true)
 |    |    |-- offset: long (nullable = true)
 |-- results: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attributes: string (nullable = true)
 |    |    |-- datatype: string (nullable = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- station: string (nullable = true)
 |    |    |-- value: double (nullable = true)



Aquí ya tenemos un ejemplo mucho más interesante de JSON anidado que los datos simples que vimos en fundamentos.

## 16. Expandir `results`

Ahora transformamos únicamente la estructura necesaria para poder inspeccionarla.

Todavía no estamos limpiando ni modificando valores.

In [24]:
weather_df = (
    weather_raw_df
    .select(
        F.explode("results").alias("record")
    )
    .select(
        "record.*"
    )
)

Y:

In [25]:
weather_df.printSchema()

root
 |-- attributes: string (nullable = true)
 |-- datatype: string (nullable = true)
 |-- date: string (nullable = true)
 |-- station: string (nullable = true)
 |-- value: double (nullable = true)



Después:

In [26]:
weather_df.show(
    n=20,
    truncate=False
)

+----------+--------+-------------------+-----------------+-----+
|attributes|datatype|date               |station          |value|
+----------+--------+-------------------+-----------------+-----+
|,,W,      |AWND    |2024-01-01T00:00:00|GHCND:USW00094728|1.5  |
|,,W,2400  |PRCP    |2024-01-01T00:00:00|GHCND:USW00094728|0.8  |
|,,W,      |SNOW    |2024-01-01T00:00:00|GHCND:USW00094728|0.0  |
|,,W,2400  |TMAX    |2024-01-01T00:00:00|GHCND:USW00094728|8.3  |
|,,W,2400  |TMIN    |2024-01-01T00:00:00|GHCND:USW00094728|1.7  |
|,,W,      |AWND    |2024-01-02T00:00:00|GHCND:USW00094728|1.8  |
|,,W,2400  |PRCP    |2024-01-02T00:00:00|GHCND:USW00094728|0.0  |
|,,W,      |SNOW    |2024-01-02T00:00:00|GHCND:USW00094728|0.0  |
|,,W,2400  |TMAX    |2024-01-02T00:00:00|GHCND:USW00094728|5.6  |
|,,W,2400  |TMIN    |2024-01-02T00:00:00|GHCND:USW00094728|-1.6 |
|,,W,      |AWND    |2024-01-03T00:00:00|GHCND:USW00094728|2.2  |
|,,W,2400  |PRCP    |2024-01-03T00:00:00|GHCND:USW00094728|0.0  |
|,,W,     

## 17. Número de registros

In [27]:
weather_rows = weather_df.count()

print(
    f"Observaciones meteorológicas: "
    f"{weather_rows:,}"
)

Observaciones meteorológicas: 1,826


## 18. Variables disponibles

In [28]:
(
    weather_df
    .groupBy("datatype")
    .count()
    .orderBy("datatype")
    .show()
)

+--------+-----+
|datatype|count|
+--------+-----+
|    AWND|  362|
|    PRCP|  366|
|    SNOW|  366|
|    TMAX|  366|
|    TMIN|  366|
+--------+-----+



## 19. Validación temporal

Primero convertimos temporalmente la fecha

In [29]:
weather_dates_df = (
    weather_df
    .withColumn(
        "observation_date",
        F.to_date("date")
    )
)

In [30]:
weather_dates_df.select(
    F.min("observation_date").alias("min_date"),
    F.max("observation_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2024-01-01|2024-12-31|
+----------+----------+



## 20. Número de días distintos

In [31]:
distinct_days = (
    weather_dates_df
    .select("observation_date")
    .distinct()
    .count()
)

print(
    f"Días meteorológicos disponibles: "
    f"{distinct_days}"
)

Días meteorológicos disponibles: 366


## 21. Controles de calidad iniciales

Una vez completada la ingesta, se realizarán controles básicos para verificar:

- cobertura temporal;
- completitud por variable meteorológica;
- fechas faltantes;
- posibles duplicados;
- consistencia de la estación;
- valores nulos.

Estos controles no modificarán los datos originales almacenados en la capa
`raw`. Su objetivo es documentar problemas que deberán considerarse durante
las transformaciones posteriores.

Primero vamos a expresar la completitud explícitamente:

In [32]:
EXPECTED_DAYS = 366

datatype_completeness_df = (
    weather_dates_df
    .groupBy("datatype")
    .agg(
        F.count("*").alias("observations"),
        F.countDistinct("observation_date").alias("distinct_days")
    )
    .withColumn(
        "expected_days",
        F.lit(EXPECTED_DAYS)
    )
    .withColumn(
        "missing_days",
        F.col("expected_days") - F.col("distinct_days")
    )
    .withColumn(
        "completeness_pct",
        F.round(
            F.col("distinct_days")
            / F.col("expected_days")
            * 100,
            2
        )
    )
    .orderBy("datatype")
)

datatype_completeness_df.show(
    truncate=False
)

+--------+------------+-------------+-------------+------------+----------------+
|datatype|observations|distinct_days|expected_days|missing_days|completeness_pct|
+--------+------------+-------------+-------------+------------+----------------+
|AWND    |362         |362          |366          |4           |98.91           |
|PRCP    |366         |366          |366          |0           |100.0           |
|SNOW    |366         |366          |366          |0           |100.0           |
|TMAX    |366         |366          |366          |0           |100.0           |
|TMIN    |366         |366          |366          |0           |100.0           |
+--------+------------+-------------+-------------+------------+----------------+



## 22. Encontrar los cuatro días faltantes de `AWND`

Generamos primero un calendario completo de 2024 utilizando Spark:

In [33]:
calendar_df = (
    spark.sql(
        """
        SELECT explode(
            sequence(
                to_date('2024-01-01'),
                to_date('2024-12-31'),
                interval 1 day
            )
        ) AS observation_date
        """
    )
)

calendar_df.show(5)

+----------------+
|observation_date|
+----------------+
|      2024-01-01|
|      2024-01-02|
|      2024-01-03|
|      2024-01-04|
|      2024-01-05|
+----------------+
only showing top 5 rows


Comprobamos:

In [34]:
print(
    f"Días generados en calendario: "
    f"{calendar_df.count()}"
)

Días generados en calendario: 366


Ahora aislamos los días disponibles para viento:

In [35]:
awnd_dates_df = (
    weather_dates_df
    .filter(
        F.col("datatype") == "AWND"
    )
    .select("observation_date")
    .distinct()
)

Y utilizamos un `left_anti` join:

In [36]:
missing_awnd_dates_df = (
    calendar_df
    .join(
        awnd_dates_df,
        on="observation_date",
        how="left_anti"
    )
    .orderBy("observation_date")
)

missing_awnd_dates_df.show(
    truncate=False
)

+----------------+
|observation_date|
+----------------+
|2024-01-24      |
|2024-01-27      |
|2024-06-23      |
|2024-06-24      |
+----------------+



## Verificar duplicados

Ahora comprobemos que no existan varias observaciones para la misma:

estación + fecha + variable

In [37]:
duplicate_weather_df = (
    weather_df
    .groupBy(
        "station",
        "date",
        "datatype"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = duplicate_weather_df.count()

print(
    f"Combinaciones duplicadas "
    f"(station, date, datatype): "
    f"{duplicate_count:,}"
)

Combinaciones duplicadas (station, date, datatype): 0


## 24. Validar estación

In [38]:
weather_df.select(
    "station"
).distinct().show(
    truncate=False
)

+-----------------+
|station          |
+-----------------+
|GHCND:USW00094728|
+-----------------+



In [39]:
station_count = (
    weather_df
    .select("station")
    .distinct()
    .count()
)

print(
    f"Número de estaciones presentes: "
    f"{station_count}"
)

Número de estaciones presentes: 1


## 25. Revisar valores nulos

In [40]:
weather_df.select(
    [
        F.sum(
            F.col(column).isNull().cast("int")
        ).alias(column)
        for column in weather_df.columns
    ]
).show(
    vertical=True,
    truncate=False
)

-RECORD 0---------
 attributes | 0   
 datatype   | 0   
 date       | 0   
 station    | 0   
 value      | 0   



## 26. Explorar rangos de las variables

No estamos haciendo todavía el EDA completo, pero sí deberíamos validar que los valores tengan magnitudes razonables.

In [41]:
weather_range_df = (
    weather_df
    .groupBy("datatype")
    .agg(
        F.count("value").alias("count"),
        F.min("value").alias("min"),
        F.avg("value").alias("mean"),
        F.max("value").alias("max")
    )
    .orderBy("datatype")
)

weather_range_df.show(
    truncate=False
)

+--------+-----+-----+------------------+----+
|datatype|count|min  |mean              |max |
+--------+-----+-----+------------------+----+
|AWND    |362  |0.4  |2.188121546961326 |6.3 |
|PRCP    |366  |0.0  |3.2185792349726774|93.0|
|SNOW    |366  |0.0  |0.7131147540983607|81.0|
|TMAX    |366  |-6.0 |18.317759562841527|35.0|
|TMIN    |366  |-10.5|10.537431693989069|26.1|
+--------+-----+-----+------------------+----+



## 27. Control de temperaturas

Sí podemos realizar una validación estructural bastante evidente:

> La temperatura mínima diaria no debería ser superior a la máxima diaria.

Aquí no necesitamos hacer todavía el pivot definitivo. Podemos crear un DataFrame temporal exclusivamente para control de calidad:

In [42]:
temperature_check_df = (
    weather_dates_df
    .filter(
        F.col("datatype").isin(
            "TMAX",
            "TMIN"
        )
    )
    .groupBy("observation_date")
    .pivot(
        "datatype",
        ["TMIN", "TMAX"]
    )
    .agg(
        F.first("value")
    )
)

In [43]:
invalid_temperature_df = (
    temperature_check_df
    .filter(
        F.col("TMIN") > F.col("TMAX")
    )
)

invalid_temperature_count = (
    invalid_temperature_df.count()
)

print(
    f"Días con TMIN > TMAX: "
    f"{invalid_temperature_count}"
)

Días con TMIN > TMAX: 0


## 28. Completitud global

Vamos a dejar también una métrica general:

In [44]:
expected_observations = (
    EXPECTED_DAYS
    * len(WEATHER_VARIABLES)
)

actual_observations = (
    weather_df.count()
)

missing_observations = (
    expected_observations
    - actual_observations
)

global_completeness = (
    actual_observations
    / expected_observations
    * 100
)

print(
    f"Observaciones esperadas: "
    f"{expected_observations:,}"
)

print(
    f"Observaciones obtenidas: "
    f"{actual_observations:,}"
)

print(
    f"Observaciones faltantes: "
    f"{missing_observations:,}"
)

print(
    f"Completitud global: "
    f"{global_completeness:.2f}%"
)

Observaciones esperadas: 1,830
Observaciones obtenidas: 1,826
Observaciones faltantes: 4
Completitud global: 99.78%


## 29. Resumen de la ingesta NOAA


In [45]:
weather_summary_data = [
    (
        "Dataset",
        DATASET_ID
    ),
    (
        "Estación",
        STATION_ID
    ),
    (
        "Año",
        str(YEAR)
    ),
    (
        "Páginas descargadas",
        str(len(weather_files))
    ),
    (
        "Observaciones esperadas",
        f"{expected_observations:,}"
    ),
    (
        "Observaciones obtenidas",
        f"{actual_observations:,}"
    ),
    (
        "Observaciones faltantes",
        f"{missing_observations:,}"
    ),
    (
        "Completitud",
        f"{global_completeness:.2f}%"
    ),
    (
        "Días disponibles",
        str(distinct_days)
    ),
    (
        "Variables",
        str(len(WEATHER_VARIABLES))
    ),
    (
        "Formato raw",
        "JSON"
    )
]

weather_summary_df = (
    spark.createDataFrame(
        weather_summary_data,
        [
            "metric",
            "value"
        ]
    )
)

weather_summary_df.show(
    truncate=False
)

+-----------------------+-----------------+
|metric                 |value            |
+-----------------------+-----------------+
|Dataset                |GHCND            |
|Estación               |GHCND:USW00094728|
|Año                    |2024             |
|Páginas descargadas    |2                |
|Observaciones esperadas|1,830            |
|Observaciones obtenidas|1,826            |
|Observaciones faltantes|4                |
|Completitud            |99.78%           |
|Días disponibles       |366              |
|Variables              |5                |
|Formato raw            |JSON             |
+-----------------------+-----------------+



## 30. Conclusiones

La ingesta de información meteorológica de NOAA para 2024 se completó
correctamente mediante la API Climate Data Online.

Se obtuvieron 1,826 observaciones correspondientes a cinco variables
meteorológicas de la estación NY City Central Park, cubriendo los 366 días
del año 2024.

Los principales controles de calidad mostraron:

- `PRCP`, `SNOW`, `TMAX` y `TMIN` presentan cobertura completa de 366 días.
- `AWND` presenta 362 observaciones, con cuatro fechas ausentes:
  - 2024-01-24
  - 2024-01-27
  - 2024-06-23
  - 2024-06-24
- No se detectaron registros duplicados para la combinación
  `(station, date, datatype)`.
- No se encontraron valores nulos en las observaciones recibidas.
- No existen días donde la temperatura mínima sea superior a la máxima.
- La completitud global del conjunto meteorológico es de aproximadamente
  99.78 %.

Las respuestas originales se conservaron sin modificar en la capa `raw`.
Las cuatro observaciones faltantes de viento serán evaluadas posteriormente
durante las etapas de transformación e integración.

Con esta etapa, el proyecto dispone de dos fuentes heterogéneas:

- NYC Yellow Taxi: archivos Parquet mensuales.
- NOAA GHCND: API REST con respuestas JSON paginadas.